In [1]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator  # type: ignore
from tensorflow.keras.models import Sequential  # type: ignore
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization # type: ignore
from tensorflow.keras.optimizers import Adam # type: ignore
from tensorflow.keras.models import load_model # type: ignore
import cv2
import numpy as np

2025-04-05 15:27:51.898738: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/Users/raghava/Documents/Projects/4-Report Generator and Load Runner/report-automation-website/lib/python3.9/site-packages/urllib3/__init__.py:34: NotOpenSSLWarning: urllib3 v2.0 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:

data_dir = "/Users/raghava/Documents/Projects/4-Report Generator and Load Runner/charts_classification_training_images"
classes = ["leak", "spikes", "no_issue"]
model_name = "chart_classification_model.h5"

In [3]:
# Image size close to original aspect ratio (2:1 ratio like 1783x883)
img_height, img_width = 448, 224

datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    validation_split=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
)

train_generator = datagen.flow_from_directory(
    data_dir,
    target_size=(img_height, img_width),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_generator = datagen.flow_from_directory(
    data_dir,
    target_size=(img_height, img_width),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

model = Sequential([
    Conv2D(16, (3, 3), activation='relu', input_shape=(img_height, img_width, 3)),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Conv2D(32, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Dropout(0.3),
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')
])


# Compile the model with a lower learning rate for better convergence
model.compile(optimizer=Adam(learning_rate=0.0005), loss='categorical_crossentropy', metrics=['accuracy'])


Found 480 images belonging to 3 classes.
Found 120 images belonging to 3 classes.


/Users/raghava/Documents/Projects/4-Report Generator and Load Runner/report-automation-website/lib/python3.9/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [4]:
weights = model.get_weights()  # returns a list of all weight arrays
for i, w in enumerate(weights[:5]):
    print(f"\nWeight {i} — shape: {w.shape}")
    print(w)



Weight 0 — shape: (3, 3, 3, 16)
[[[[ 0.10082543  0.0462911  -0.14447132  0.02709766  0.09470665
     0.01632899 -0.16529569 -0.02757664 -0.1275315   0.0516388
     0.10481498 -0.131011    0.09360337 -0.00695957 -0.1757628
     0.18252024]
   [ 0.02408205  0.1568658   0.04945083 -0.08316985 -0.01581872
     0.17866439  0.09274983 -0.0349955   0.01963249  0.1375868
    -0.0605765  -0.08713788 -0.06973942 -0.00491844 -0.15233748
     0.01403366]
   [ 0.1561569   0.03327069  0.11722776 -0.07435193 -0.1103432
     0.01775078 -0.10414785 -0.09923071 -0.1209502   0.07740024
     0.04469107 -0.13342947 -0.06618726  0.13662064 -0.13091119
     0.01178557]]

  [[-0.10209966 -0.17729896  0.1688807   0.17117953 -0.14476308
     0.06875092  0.02632576 -0.1390236  -0.13120933 -0.18012598
    -0.00748843 -0.03676756  0.06345153 -0.17476255 -0.06741822
    -0.07644809]
   [-0.03134356 -0.12437347  0.13785157  0.06416041 -0.06446753
     0.04395038  0.10089821  0.00719538 -0.06018519  0.0646866
    -0

In [5]:

# Train the model
model.fit(train_generator, validation_data=val_generator, epochs=20)


/Users/raghava/Documents/Projects/4-Report Generator and Load Runner/report-automation-website/lib/python3.9/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 57s 4s/step - accuracy: 0.5961 - loss: 4.9476 - val_accuracy: 0.3333 - val_loss: 5.3047
Epoch 2/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 47s 3s/step - accuracy: 0.9094 - loss: 0.6377 - val_accuracy: 0.3333 - val_loss: 12.0838
Epoch 3/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 50s 3s/step - accuracy: 0.9611 - loss: 0.3296 - val_accuracy: 0.3333 - val_loss: 17.0117
Epoch 4/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 48s 3s/step - accuracy: 0.9696 - loss: 0.2223 - val_accuracy: 0.3333 - val_loss: 19.7333
Epoch 5/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 50s 3s/step - accuracy: 0.9562 - loss: 0.1885 - val_accuracy: 0.3333 - val_loss: 25.0574
Epoch 6/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 46s 3s/step - accuracy: 0.9687 - loss: 0.1706 - val_accuracy: 0.3333 - val_loss: 28.5111
Epoch 7/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 44s 3s/step - accuracy: 0.9733 - loss: 0.1599 - val_accuracy: 0.3333 - val_loss: 28.7617
Epoch 8/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 45s 3s/step - accuracy: 0.9770 - loss: 0.1299 - val_accuracy: 0.3333 - val

In [ ]:
weights = model.get_weights()  # returns a list of all weight arrays
for i, w in enumerate(weights[:5]):
    print(f"\nWeight {i} — shape: {w.shape}")
    print(w)


In [11]:

    
# Save the model
model.save(model_name)

In [7]:
weights = model.get_weights()  # returns a list of all weight arrays
for i, w in enumerate(weights[:5]):
    print(f"\nWeight {i} — shape: {w.shape}")
    print(w)



Weight 0 — shape: (3, 3, 3, 16)
[[[[ 1.04338013e-01  4.78915647e-02 -1.60807773e-01  3.16723362e-02
     9.23135132e-02 -4.60960809e-03 -1.61218584e-01 -3.08355764e-02
    -1.29622236e-01  4.15425412e-02  9.84270573e-02 -1.40046790e-01
     8.39143619e-02 -1.84582435e-02 -1.73843741e-01  2.09072828e-01]
   [ 2.75880210e-02  1.57527193e-01  4.64617498e-02 -8.20624381e-02
    -1.81812905e-02  1.69492051e-01  9.37599316e-02 -4.09925543e-02
     1.61110200e-02  1.26782686e-01 -5.98945208e-02 -9.13322717e-02
    -7.94925466e-02 -8.28154664e-03 -1.50866166e-01  3.39440070e-02]
   [ 1.59608871e-01  3.29198986e-02  1.35570496e-01 -7.58723915e-02
    -1.12644322e-01  2.87218727e-02 -1.04242876e-01 -1.07680991e-01
    -1.24136172e-01  6.60334155e-02  5.32931015e-02 -1.33476689e-01
    -7.61235654e-02  1.42342076e-01 -1.30281895e-01  2.21809857e-02]]

  [[-9.87368003e-02 -1.68560520e-01  1.55260384e-01  1.83540806e-01
    -1.47869974e-01  4.42691930e-02  2.49965116e-02 -1.42527610e-01
    -1.358

In [25]:
# Set your image dimensions here (must match model input)
img_height, img_width = 448, 224

def predict(image_path, loaded_model):
    def preprocess_image(image_path):
        img = cv2.imread(image_path)  # Load the image
        # if img is None:
        #     return f"cv2 failed to read the image. Path exists? {os.path.exists(image_path)}. path : {image_path} Exact path: {repr(image_path)}"
        # else:
            # return f"Image loaded successfully. Shape: {img.shape}"
        img = cv2.resize(img, (img_width, img_height))  # Resize to (width, height)
        img = img / 255.0  # Normalize pixel values
        img = np.expand_dims(img, axis=0)  # Add batch dimension
        return img

    processed_image = preprocess_image(image_path)

    prediction = loaded_model.predict(processed_image)
    class_index = np.argmax(prediction)

    predicted_label = classes[class_index]

    return_string = f"Predicted Class: {predicted_label}.\nConfidence {prediction[0][class_index]*100:.2f}%\n"
    # for i, label in enumerate(classes):
    #     return_string += f"{label}: {prediction[0][i]*100:.2f}% confidence\n"

    return return_string


In [26]:
loaded_model = load_model(model_name)
print(predict("/Users/raghava/Documents/Projects/graphs/Osquery_LoadTests_New/MultiCustomer/67c5831ed8a2dee35f06d25b/Live Assets and All Lags/Live Assets Count.png", loaded_model))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 383ms/step
Predicted Class: leak.
Confidence 100.00%



In [15]:
weights = loaded_model.get_weights()  # returns a list of all weight arrays
for i, w in enumerate(weights[:5]):
    print(f"\nWeight {i} — shape: {w.shape}")
    print(w)



Weight 0 — shape: (3, 3, 3, 16)
[[[[ 1.04338013e-01  4.78915647e-02 -1.60807773e-01  3.16723362e-02
     9.23135132e-02 -4.60960809e-03 -1.61218584e-01 -3.08355764e-02
    -1.29622236e-01  4.15425412e-02  9.84270573e-02 -1.40046790e-01
     8.39143619e-02 -1.84582435e-02 -1.73843741e-01  2.09072828e-01]
   [ 2.75880210e-02  1.57527193e-01  4.64617498e-02 -8.20624381e-02
    -1.81812905e-02  1.69492051e-01  9.37599316e-02 -4.09925543e-02
     1.61110200e-02  1.26782686e-01 -5.98945208e-02 -9.13322717e-02
    -7.94925466e-02 -8.28154664e-03 -1.50866166e-01  3.39440070e-02]
   [ 1.59608871e-01  3.29198986e-02  1.35570496e-01 -7.58723915e-02
    -1.12644322e-01  2.87218727e-02 -1.04242876e-01 -1.07680991e-01
    -1.24136172e-01  6.60334155e-02  5.32931015e-02 -1.33476689e-01
    -7.61235654e-02  1.42342076e-01 -1.30281895e-01  2.21809857e-02]]

  [[-9.87368003e-02 -1.68560520e-01  1.55260384e-01  1.83540806e-01
    -1.47869974e-01  4.42691930e-02  2.49965116e-02 -1.42527610e-01
    -1.358